# 雪乃 · A100 纠错硬标签续训 v3

从云盘中已经完成的 v1 人格 LoRA 继续训练同一 **Qwen2.5-1.5B-Instruct**。
本次由 Codex 依据小说和角色档案编写 192 组原创标准回答，明确标记为合成数据：
144 组用于训练、24 组验证、24 组最终检查。混入原来仅训练划分的 186 组人格问答，减轻遗忘。
不需要教师 API key，也不会假称已经调用其他在线模型。

例如：阳乃是姐姐，雪乃还有父亲和母亲；独居不能推出没有其他家人。
每个事实场景有来源编号，训练只计算最后一条正确回答的损失。

**顺序运行 1–5；第 6 节导出 GGUF；第 7 节是可连续/批量提问的并排界面。**
上限 12 轮，每轮完整保存；连续 3 轮改善不足 0.001 时早停，最终选择最低验证损失的断点。
最终测试不用于早停或选择模型。轮次和损失无法保证没有幻觉，最后仍要读实际回答。

所有代码快照、原始资料、标签、配置、基座、断点、日志和最终模型都在 Drive。
`/content/yukino-distill-v3-runtime` 只放可重建依赖与编译缓存。

## 1. 挂载云盘与固定路径

先在「更改运行时类型」选择 A100。`RUN_ID` 固定一个实验；断线后保持相同值并从第 1 节重跑。
如果提示找不到 `nvidia-smi`，先检查「代码执行程序 → 更改运行时类型 → A100 GPU」，保存并重新连接。
代码会用 PyTorch 再确认实际 GPU；无需在 CPU 运行时安装 NVIDIA 驱动。云盘资料不受本检测影响。
改标签、学习率或总轮数必须使用新 `RUN_ID`，防止新旧训练状态混用。
`ATTEMPT` 只用于第一次保存前中断、没有可恢复断点时重新开一个训练目录。

In [ ]:
import json, os, re, shutil, subprocess, sys, tempfile
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
from google.colab import drive

# 实际 CUDA 设备决定能否训练；笔记本名称或元数据不会分配 GPU。
def require_a100():
    import shutil
    import subprocess
    from pathlib import Path

    help_text = ("请打开「代码执行程序（Runtime）→ 更改运行时类型 → A100 GPU」，"
                 "保存并重新连接，然后从第 1 节运行。只写 A100 配置不会自动获得 GPU。")
    smi = shutil.which("nvidia-smi") or next(
        (p for p in ("/usr/bin/nvidia-smi", "/usr/local/nvidia/bin/nvidia-smi") if Path(p).is_file()), None)
    if smi:
        try:
            print(subprocess.check_output([smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                          text=True, stderr=subprocess.STDOUT, timeout=15).strip())
        except (OSError, subprocess.SubprocessError) as error:
            print("nvidia-smi 诊断不可用，将使用 PyTorch 检查实际 GPU：", type(error).__name__)
    else:
        print("未找到 nvidia-smi，正在检查 PyTorch 是否能访问 CUDA GPU。")
    try:
        import torch
        if not torch.cuda.is_available():
            raise SystemExit("当前没有可用的 CUDA GPU，训练尚未开始。" + help_text)
        name = torch.cuda.get_device_name(0)
        if "A100" not in name.upper():
            raise SystemExit(f"当前实际分配的是 {name}，本笔记本要求 A100。" + help_text)
        if not torch.cuda.is_bf16_supported():
            raise SystemExit("当前 CUDA 环境不支持本训练所需的 BF16。" + help_text)
        print("CUDA 检查通过：", name)
        return name
    except Exception as error:
        raise SystemExit("无法初始化当前运行时的 CUDA/PyTorch：" + str(error) + "。" + help_text) from None

gpu = require_a100()
drive.mount("/content/drive")
RUN_ID = "epub-a100-v3-distill"
MAX_EPOCHS = 12
EARLY_STOP_PATIENCE = 3
ATTEMPT = 1
INITIAL_RUN = "epub-a100-v1"
INITIAL_STAGE = "persona-01"
for value in (RUN_ID, INITIAL_RUN, INITIAL_STAGE):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", value):
        raise ValueError("实验和阶段名只能使用英文字母、数字、横线和下划线。")
if type(MAX_EPOCHS) is not int or not 2 <= MAX_EPOCHS <= 50 or type(EARLY_STOP_PATIENCE) is not int or not 1 <= EARLY_STOP_PATIENCE <= MAX_EPOCHS:
    raise ValueError("轮次范围 2–50，早停耐心值必须在 1 到总轮次之间。")
if type(ATTEMPT) is not int or ATTEMPT < 1:
    raise ValueError("ATTEMPT 必须为正整数。")
project = Path("/content/drive/MyDrive/Yukino-Companion")
if not project.is_dir():
    raise FileNotFoundError("未找到已上传的 Yukino-Companion 云盘目录。")
root = Path("/content/yukino-distill-v3-runtime")
root.mkdir(exist_ok=True)
drive_runs = project / "retrain" / RUN_ID
drive_runs.mkdir(parents=True, exist_ok=True)
epubs = project / "data/epubs"
if not list(epubs.glob("*.epub")):
    raise FileNotFoundError(epubs)
initial_source = project / "retrain" / INITIAL_RUN / INITIAL_STAGE
persona_output = drive_runs / f"distill-{ATTEMPT:02d}"
base_revision = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

def digest(path):
    h = sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)

def inventory(directory):
    return {p.relative_to(directory).as_posix(): digest(p) for p in sorted(Path(directory).rglob("*")) if p.is_file() and p.name != "artifacts.json"}

def publish_directory(local, final):
    final.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".publish-", dir=final.parent) as staging:
        candidate = Path(staging) / final.name
        shutil.copytree(local, candidate)
        if inventory(candidate) != inventory(local):
            raise RuntimeError("复制后的 Drive 文件校验失败。")
        candidate.rename(final)

def verify_run(output, task):
    report = load_json(output / "training_report.json")
    if not (report.get("completed") and report.get("run_purpose") == "training"
            and report.get("task") == task and report.get("base_revision") == base_revision
            and report.get("device") == "cuda" and report.get("dtype") == "torch.bfloat16"
            and report.get("steps", 0) > 1
            and report.get("adapter_sha256") == digest(output / "adapter/adapter_model.safetensors")):
        raise RuntimeError(f"模型报告或权重校验失败：{output}")
    return report

initial_report = verify_run(initial_source, "complete_persona")
print("继续训练来源", initial_source, initial_report["adapter_sha256"])
print("持久化输出", drive_runs)


## 2. 从云盘恢复代码与依赖

源码 ZIP 和解压后的代码都保存在 `Yukino-Companion/source/`。
依赖环境借用 Colab 已有 CUDA Torch，不再安装容易失败的旧 Torch wheel。
基础模型保存到 `Yukino-Companion/models/`；可从同一运行时的旧缓存复制，缺失时由 Colab 服务器直接下载。

In [ ]:
import tomllib
source_archive = project / "source/training-source-distill-v3.zip"
source_hash = "2c31dae156f74497a7a5fa8a60341baa1b12b4d5aeb7d944fd09461ca82403ea"
if not source_archive.is_file() or digest(source_archive) != source_hash:
    raise RuntimeError(f"源码归档缺失或 SHA 不符：{source_archive}")
repo = project / "source" / ("distill-v3-" + source_hash[:12])
with ZipFile(source_archive) as archive:
    for item in archive.infolist():
        if item.filename.startswith("/") or ".." in Path(item.filename).parts:
            raise ValueError("源码归档存在不安全路径。")
    if not repo.exists():
        with tempfile.TemporaryDirectory(prefix=".source-", dir=repo.parent) as stage:
            candidate = Path(stage) / "source"
            archive.extractall(candidate)
            candidate.rename(repo)
    for item in archive.infolist():
        if not item.is_dir() and (repo / item.filename).read_bytes() != archive.read(item):
            raise RuntimeError("云盘源码快照被改动：" + item.filename)
service = repo / "model-service"
venv = root / "venv"
python = venv / "bin/python"
if not python.exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(venv)], check=True)
def run(*args, cwd=service, log=None):
    command = [str(python), *(str(a) for a in args)]
    env = dict(os.environ, USE_TF="0", USE_TORCH="1", PYTHONPATH=str(service), PYTHONUNBUFFERED="1")
    log = Path(log or drive_runs / "operations.log")
    with log.open("a", encoding="utf-8", buffering=1) as stream:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
run("-c", "import torch; print(torch.__version__, torch.version.cuda); assert tuple(map(int,torch.__version__.split('.')[:2])) >= (2,6); assert 'A100' in torch.cuda.get_device_name(0) and torch.cuda.is_bf16_supported(); print(torch.ones(1,device='cuda',dtype=torch.bfloat16))")
config = tomllib.loads((service / "pyproject.toml").read_text(encoding="utf-8"))["project"]
dependencies = config["dependencies"] + [s for s in config["optional-dependencies"]["train"] if not s.startswith("torch")]
ready = venv / "deps-ready"
if not ready.exists():
    run("-m", "pip", "install", "--disable-pip-version-check", "--retries", "3", "--timeout", "60", *dependencies)
run("-c", "from transformers import Trainer; from peft import PeftModel; from yuki_model.distillation import build_distillation")
ready.touch()
run("-m", "pip", "list", "--format=json", log=drive_runs / "environment.log")
base = project / "models/qwen2.5-1.5b-instruct-989aa798"
if not (base / "BASE_PROVENANCE.json").is_file():
    old_cache = Path("/content/yukino-a100/source/model-service/models/base")
    if not base.exists() and (old_cache / "BASE_PROVENANCE.json").is_file():
        publish_directory(old_cache, base)
    else:
        run("-m", "yuki_model", "download", "--output", base)
base_proof = load_json(base / "BASE_PROVENANCE.json")
if base_proof["base_revision"] != base_revision:
    raise RuntimeError("基座版本不匹配。")
for name, expected in base_proof["files"].items():
    if Path(name).name != name or digest(base / name) != expected:
        raise RuntimeError("基座文件校验失败：" + name)
if digest(base / "model.safetensors") != "dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee":
    raise RuntimeError("基座权重不符合固定版本。")
print("代码路径", repo)
print("基座路径", base)


## 3. 核验小说证据，保存标签与旧权重快照

核验 26 条出处的文件 SHA 和原文段落；生成 330 组训练、24 组验证和 24 组最终检查。
训练仅使用旧人格数据的训练部分；本次测试题的答案不会放入评测提示。
12 道已训练的纠错题另存为回归检查，不能把这些成绩当成未见题表现。
新问法仍围绕熟悉事实，不代表模型掌握所有原作内容。

In [ ]:
inputs = {
    "source_archive_sha256": source_hash, "base_revision": base_revision,
    "initial_path": str(initial_source), "initial_sha256": initial_report["adapter_sha256"],
    "initial_config_sha256": digest(initial_source / "adapter/adapter_config.json"),
    "epochs": MAX_EPOCHS, "patience": EARLY_STOP_PATIENCE,
    "epubs": {p.name: digest(p) for p in sorted(epubs.glob("*.epub"))},
}
input_file = drive_runs / "inputs.json"
if input_file.exists() and load_json(input_file) != inputs:
    raise RuntimeError("该 RUN_ID 已绑定另一份输入或参数，请换新 RUN_ID。")
save_json(input_file, inputs)
initial_snapshot = drive_runs / "initial"
if not initial_snapshot.exists():
    with tempfile.TemporaryDirectory(prefix=".initial-", dir=drive_runs) as stage:
        candidate = Path(stage) / "initial"
        candidate.mkdir()
        shutil.copytree(initial_source / "adapter", candidate / "adapter")
        shutil.copyfile(initial_source / "training_report.json", candidate / "training_report.json")
        if digest(candidate / "adapter/adapter_model.safetensors") != inputs["initial_sha256"]:
            raise RuntimeError("旧权重快照复制校验失败。")
        candidate.rename(initial_snapshot)
verify_run(initial_snapshot, "complete_persona")
if digest(initial_snapshot / "adapter/adapter_config.json") != inputs["initial_config_sha256"]:
    raise RuntimeError("旧适配器配置与快照不符。")
prepared = drive_runs / "prepared"
if not prepared.exists():
    with tempfile.TemporaryDirectory(prefix=".prepare-", dir=drive_runs) as stage:
        candidate = Path(stage) / "data"
        run("scripts/build_distillation_sft.py", "--epubs", epubs, "--tokenizer", base,
            "--output", candidate, "--sequence-length", "1536", log=drive_runs / "prepare.log")
        candidate.rename(prepared)
data_manifest = load_json(prepared / "manifest.json")
for name, expected in data_manifest["artifacts"].items():
    if Path(name).name != name or digest(prepared / name) != expected:
        raise RuntimeError("生成数据被修改，请新建实验：" + name)
if data_manifest["source_sha256"] != digest(service / "distill-data/seeds.jsonl") or data_manifest["evidence_sha256"] != digest(service / "distill-data/evidence.jsonl"):
    raise RuntimeError("数据与源码快照中的标签或证据不一致。")
print("问答数量", data_manifest["examples"])
print("训练来源", data_manifest["origin_counts"]["train"])
print("最大 tokens", data_manifest["max_tokens"])
print("待人工检查的近似问法", data_manifest["near_prompt_pairs_for_review"])


## 4. 继续微调：最多 12 轮，每轮保存，自动续接

从第 3 节冻结的旧人格 LoRA 继续更新，不重新训练旧阶段。
每轮保存适配器、优化器、调度器、随机状态、早停状态和 SHA 完成标记；
中断时跳过未写完的断点，续接最近一份完整状态。
保存全部已完成轮次，最终 `adapter/` 是验证损失最小的一轮。
第一次保存前中断时会明确要求增加 `ATTEMPT`，不会悄悄覆盖数据。

In [ ]:
if "stop_comparison_worker" in globals():
    stop_comparison_worker()
config = load_json(service / "configs/train-distill.json")
config.update(model=str(base), initial_adapter=str(initial_snapshot / "adapter"),
              train_file=str(prepared / "train.jsonl"), eval_file=str(prepared / "val.jsonl"),
              data_manifest=str(prepared / "manifest.json"), output_dir=str(persona_output),
              epochs=MAX_EPOCHS, save_total_limit=MAX_EPOCHS,
              early_stopping_patience=EARLY_STOP_PATIENCE)
config_file = drive_runs / f"{persona_output.name}-config.json"
if config_file.exists() and load_json(config_file) != config:
    raise RuntimeError("已有训练参数不同，请使用新的 RUN_ID。")
save_json(config_file, config)
if (persona_output / "training_report.json").is_file():
    report = verify_run(persona_output, "distill_sft")
    if report["data"]["data_manifest_sha256"] != digest(prepared / "manifest.json"):
        raise RuntimeError("已完成模型与当前数据不一致。")
    print("此阶段已经完成，读取保存结果。")
else:
    resume = persona_output.exists() and any(persona_output.iterdir())
    runner = '''
import json, sys
from pathlib import Path
from yuki_model.training import train, validate_resume, validate_training_spec
from yuki_model.io import file_sha
c = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
if sys.argv[2] == "resume":
    c["resume"] = True
    try:
        selected = validate_resume(Path(c["output_dir"]), c, {
            "train_sha256": file_sha(c["train_file"]), "eval_sha256": file_sha(c["eval_file"]),
            **validate_training_spec(c)})
    except (ValueError, OSError) as error:
        raise RuntimeError("没有可恢复的完整断点；保留原目录，增加第1节 ATTEMPT 后重跑。详情：" + str(error)) from error
    print("恢复断点", selected, flush=True)
print(json.dumps(train(c), ensure_ascii=False, indent=2))
'''
    run("-c", runner, config_file, "resume" if resume else "fresh", log=drive_runs / f"{persona_output.name}.log")
    report = verify_run(persona_output, "distill_sft")
print("实际完成轮次", report["actual_epoch"], "上限", report["configured_epochs"], "早停", report["stopped_early"])
print("导出权重来源", report["selected_weights"], report["best_checkpoint"])
print("训练前/后验证损失", (report["before"] or {}).get("eval_loss"), report["after"]["eval_loss"])
print("模型已保存在 Drive", persona_output / "adapter")


## 5. 同题比较继续训练前后：回归题与保留题

同一基座、同一角色提示和确定性解码，不注入参考答案。
每生成一条回答就保存；中断后运行本节只补完缺失部分。
`regression` 是 12 道训练内纠错题，`held-out` 是 24 道未参加本轮训练和选模的题。
报告没有自动宣称准确率；展开参考答案并结合来源读回答，CSV 可记录事实错误和偏好。

In [ ]:
from IPython.display import HTML, display
if "stop_comparison_worker" in globals():
    stop_comparison_worker()
verify_run(persona_output, "distill_sft")
for label, name in (("regression", "regression"), ("held-out", "test")):
    output = drive_runs / "evaluation" / persona_output.name / label
    run("scripts/evaluate_distillation.py", "--model", base, "--device", "cuda",
        "--before-adapter", initial_snapshot / "adapter", "--after-adapter", persona_output / "adapter",
        "--prompts", prepared / f"{name}-prompts.jsonl", "--references", prepared / f"{name}.jsonl",
        "--evaluation-kind", label, "--output", output, log=drive_runs / "evaluation.log")
    display(HTML((output / "comparison.html").read_text(encoding="utf-8")))
    print("云盘报告", output)


## 6. 导出候选离线 GGUF 模型

导出最佳验证轮次的 LoRA 为 Q4_K_M；合并 FP16 及转换中间文件也放在 Drive，
成功后仅清理本节创建的临时中间目录，最终 GGUF、来源报告与训练断点仍保留。
这是候选模型，文件完整性验证不代表回答质量合格。

In [ ]:
if "stop_comparison_worker" in globals():
    stop_comparison_worker()
# 5. 在本地合并与量化，再将校验后的候选 GGUF 保存到 Drive
EXPORT_GGUF = True  # 仅训练 LoRA 时可关闭

if EXPORT_GGUF:
    export_report = verify_run(persona_output, "distill_sft")
    export_path = drive_runs / f"{persona_output.name}-gguf"
    converter = root / "llama.cpp"
    converter_commit = "b3fa69955ea81c3b60e77e3c1e2224c4a826ffb9"

    if not export_path.exists():
        if shutil.disk_usage(drive_runs).free < 12 * 1024**3:
            raise RuntimeError("GGUF 合并与量化至少需要 12 GiB 云盘可用空间。")
        if not (converter / ".git").is_dir():
            subprocess.run(["git", "init", str(converter)], check=True)
        subprocess.run(["git", "-C", str(converter), "fetch", "--depth=1",
                        "https://github.com/ggml-org/llama.cpp.git", converter_commit], check=True)
        subprocess.run(["git", "-C", str(converter), "checkout", "--detach", "FETCH_HEAD"], check=True)
        actual = subprocess.check_output(["git", "-C", str(converter), "rev-parse", "HEAD"], text=True).strip()
        if actual != converter_commit:
            raise RuntimeError("llama.cpp 转换器版本不匹配。")
        run("-m", "pip", "install", "numpy==2.2.6", "sentencepiece==0.2.1", "protobuf==4.25.8")
        build = converter / "build"
        subprocess.run(["cmake", "-S", str(converter), "-B", str(build),
                        "-DCMAKE_BUILD_TYPE=Release", "-DGGML_NATIVE=OFF",
                        "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_TOOLS=ON",
                        "-DLLAMA_BUILD_COMMON=ON", "-DLLAMA_BUILD_APP=OFF",
                        "-DLLAMA_BUILD_SERVER=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF",
                        "-DLLAMA_OPENSSL=OFF"], check=True)
        subprocess.run(["cmake", "--build", str(build), "--target", "llama-quantize", "--parallel", "2"], check=True)
        with tempfile.TemporaryDirectory(prefix=".export-", dir=drive_runs) as work:
            work = Path(work)
            merged = work / "merged"
            candidate = work / "gguf"
            run("-m", "yuki_model", "merge", "--base", base, "--adapter", persona_output / "adapter",
                "--output", merged)
            merge_check = """
import sys
from safetensors import safe_open
import torch

key = "model.layers.0.self_attn.q_proj.weight"
with safe_open(sys.argv[1], framework="pt", device="cpu") as file:
    original = file.get_tensor(key)
with safe_open(sys.argv[2], framework="pt", device="cpu") as file:
    updated = file.get_tensor(key)
if original.shape != updated.shape or not torch.isfinite(updated).all():
    raise RuntimeError("Merged tensor is incompatible or non-finite")
changed = int(torch.count_nonzero(updated != original.to(updated.dtype)).item())
if changed == 0:
    raise RuntimeError("LoRA merge did not change the targeted tensor")
print(changed)
"""
            checked = subprocess.check_output(
                [str(python), "-c", merge_check, str(base / "model.safetensors"), str(merged / "model.safetensors")],
                text=True,
            ).strip()
            save_json(merged / "merge-validation.json", {
                "checkedTensor": "model.layers.0.self_attn.q_proj.weight",
                "changedElements": int(checked),
                "adapterSha256": export_report["adapter_sha256"],
            })
            run("scripts/export_gguf.py", "--llama-cpp", converter, "--merged", merged,
                "--training-report", persona_output / "training_report.json", "--base-license", base / "LICENSE",
                "--quantizer", build / "bin/llama-quantize", "--output", candidate,
                "--work-dir", work / "conversion", "--remove-fp16")
            manifest = load_json(candidate / "manifest.json")
            manifest["name"] = f"雪乃本地角色模型 · Qwen 1.5B · A100 · {RUN_ID}/{persona_output.name}"
            save_json(candidate / "manifest.json", manifest)
            publish_directory(candidate, export_path)

    manifest = load_json(export_path / "manifest.json")
    weights = export_path / manifest["file"]
    with weights.open("rb") as stream:
        if stream.read(4) != b"GGUF":
            raise RuntimeError("导出的文件缺少 GGUF 文件头。")
    if (manifest["training"]["adapterSha256"] != export_report["adapter_sha256"]
            or manifest["llamaCppConverterCommit"] != converter_commit
            or weights.stat().st_size != manifest["bytes"] or digest(weights) != manifest["sha256"]):
        raise RuntimeError("Drive 上的 GGUF 权重或来源校验失败。")
    print("候选 GGUF 已保存并通过文件校验，等待人工评估：", weights)
else:
    print("本次跳过 GGUF 导出。")


## 7. 双模型问答前端

左边默认为本次继续训练前的 v1，右边为纠错续训后的模型。
每行一题，最多 12 道；也可以选择连续对话，两个模型各自保留最近四轮。
输入问题时才加载 GPU 模型；开始新训练前点击释放模型。
勾选保存即可把问题、回答和权重 SHA 留在本实验的 `comparison/`。

## 7.1. 加载两份 LoRA 的推理接口

In [ ]:
import select
verify_run(persona_output, "distill_sft")
before_report = verify_run(initial_snapshot, "complete_persona")
after_report = verify_run(persona_output, "distill_sft")
comparison_models = {
    "before": {"label": "继续训练前 · " + INITIAL_RUN, "path": str(initial_snapshot / "adapter"), "sha256": before_report["adapter_sha256"]},
    "current": {"label": "纠错续训后 · " + RUN_ID, "path": str(persona_output / "adapter"), "sha256": after_report["adapter_sha256"]},
}

COMPARE_WORKER = r'''
from contextlib import redirect_stdout
import json
from pathlib import Path
import sys

with redirect_stdout(sys.stderr):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from yuki_model.persona import load_character

base = Path(sys.argv[1])
character = Path(sys.argv[2])
model = tokenizer = system = None

def reply(item):
    global model, tokenizer, system
    if model is None:
        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
            raise RuntimeError("对比需要支持 BF16 的 CUDA GPU。")
        torch.cuda.set_device(0)
        system, _ = load_character(character, "training")
        tokenizer = AutoTokenizer.from_pretrained(base, local_files_only=True, trust_remote_code=False)
        backbone = AutoModelForCausalLM.from_pretrained(
            base, torch_dtype=torch.bfloat16, local_files_only=True, trust_remote_code=False,
        ).to("cuda:0")
        model = PeftModel.from_pretrained(backbone, item["path"], adapter_name=item["key"],
                                         is_trainable=False, local_files_only=True)
    elif item["key"] not in model.peft_config:
        model.load_adapter(item["path"], adapter_name=item["key"], is_trainable=False, local_files_only=True)
    model.set_adapter(item["key"])
    model.eval()
    messages = [{"role": "system", "content": system}, *item["messages"]]
    while True:
        encoded = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        )
        length = encoded["input_ids"].shape[-1]
        if length <= 8192 - 192:
            break
        if len(messages) <= 2:
            raise ValueError("单题太长，请缩短后再试。")
        del messages[1:3]
    encoded = encoded.to("cuda:0")
    with torch.inference_mode():
        tokens = model.generate(**encoded, max_new_tokens=192, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(tokens[0, length:], skip_special_tokens=True).strip()

for line in sys.stdin:
    try:
        with redirect_stdout(sys.stderr):
            request = json.loads(line)
            answers = [reply(item) for item in request]
        response = {"answers": answers}
    except Exception as error:
        response = {"error": f"{type(error).__name__}: {error}"}
    print(json.dumps(response, ensure_ascii=False), flush=True)
'''

if "stop_comparison_worker" in globals():
    stop_comparison_worker()
comparison_worker = None
comparison_log = drive_runs / "comparison-runtime.log"

def stop_comparison_worker():
    global comparison_worker
    worker = comparison_worker
    comparison_worker = None
    if worker is None:
        return
    if worker.poll() is None:
        worker.terminate()
        try:
            worker.wait(timeout=5)
        except subprocess.TimeoutExpired:
            worker.kill()
            worker.wait()
    worker.stdin.close()
    worker.stdout.close()

def compare_request(items):
    global comparison_worker
    if comparison_worker is None or comparison_worker.poll() is not None:
        stop_comparison_worker()
        with comparison_log.open("a", encoding="utf-8") as log:
            comparison_worker = subprocess.Popen(
                [str(python), "-u", "-c", COMPARE_WORKER, str(base), str(repo / "characters/yukino")],
                cwd=service, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=log,
                text=True, encoding="utf-8", bufsize=1,
                env=dict(os.environ, USE_TF="0", USE_TORCH="1"),
            )
    worker = comparison_worker
    try:
        worker.stdin.write(json.dumps(items, ensure_ascii=False) + "\n")
        worker.stdin.flush()
        if not select.select([worker.stdout], [], [], 300)[0]:
            raise TimeoutError("推理等待超过 5 分钟。")
        line = worker.stdout.readline()
        if not line:
            raise RuntimeError(f"推理进程退出，日志见 {comparison_log}")
        response = json.loads(line)
    except (OSError, ValueError, RuntimeError) as error:
        stop_comparison_worker()
        raise RuntimeError(f"{error}。可再次点击比较，日志见 {comparison_log}") from error
    if "error" in response:
        raise RuntimeError(response["error"])
    return response["answers"]

print("可选模型", "、".join(item["label"] for item in comparison_models.values()))


## 7.2. 打开批量与连续问答界面

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

import ipywidgets as widgets
from IPython.display import display

if "compare_request" not in globals():
    raise RuntimeError("请先运行 7.1。")

model_options = [(item["label"], key) for key, item in comparison_models.items()]
left = widgets.Dropdown(options=model_options, value="before",
                        description="左栏")
right = widgets.Dropdown(options=model_options, value="current", description="右栏")
mode = widgets.ToggleButtons(options=[("每题独立", "single"), ("连续对话", "chat")])
questions = widgets.Textarea(placeholder="每行一个问题，最多 12 行", layout=widgets.Layout(width="100%", height="108px"))
ask = widgets.Button(description="比较回答", button_style="primary")
clear = widgets.Button(description="清空对话")
release_gpu = widgets.Button(description="释放模型")
save_results = widgets.Checkbox(value=False, description="保存到 Drive", indent=False)
status = widgets.Label(value="请输入问题，首次提问时加载模型。")
answers = [widgets.Textarea(disabled=True, layout=widgets.Layout(width="100%", height="360px")) for _ in range(2)]
title = [widgets.Label(), widgets.Label()]
history = [[], []]
saved_file = drive_runs / "comparison" / f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid4().hex[:6]}.jsonl"

def reset_comparison(_=None):
    history[0].clear()
    history[1].clear()
    for index, key in enumerate((left.value, right.value)):
        title[index].value = comparison_models[key]["label"]
        answers[index].value = ""
    status.value = "已清空对话。"

def release_comparison(_):
    stop_comparison_worker()
    status.value = "已释放模型，再次提问会重新加载。"

def submit_comparison(_):
    selected = (left.value, right.value)
    batch = [line.strip() for line in questions.value.splitlines() if line.strip()]
    if selected[0] == selected[1]:
        status.value = "请选择两个不同的模型。"
        return
    if not 1 <= len(batch) <= 12 or any(len(question) > 3000 for question in batch):
        status.value = "每次输入 1–12 道题，每题最多 3,000 字。"
        return
    controls = (left, right, mode, questions, ask, clear, release_gpu, save_results)
    for control in controls:
        control.disabled = True
    for box in answers:
        box.value = ""
    try:
        for number, question in enumerate(batch, 1):
            status.value = f"正在回答第 {number}/{len(batch)} 题。"
            user = {"role": "user", "content": question}
            items = [
                {"key": key, "path": comparison_models[key]["path"],
                 "messages": [*(history[index] if mode.value == "chat" else []), user]}
                for index, key in enumerate(selected)
            ]
            pair = compare_request(items)
            for index, answer in enumerate(pair):
                block = f"{number}. {question}\n\n{answer or '（模型未返回文字）'}"
                answers[index].value += ("\n\n" if answers[index].value else "") + block
                if mode.value == "chat":
                    history[index][:] = [*history[index], user, {"role": "assistant", "content": answer}][-8:]
            if save_results.value:
                saved_file.parent.mkdir(parents=True, exist_ok=True)
                record = {
                    "time": datetime.now(timezone.utc).isoformat(),
                    "mode": mode.value, "question": question,
                    "answers": [
                        {"model": comparison_models[key]["label"],
                         "adapter_sha256": comparison_models[key]["sha256"], "answer": answer}
                        for key, answer in zip(selected, pair)
                    ],
                }
                with saved_file.open("a", encoding="utf-8") as stream:
                    stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        status.value = f"已回答 {len(batch)} 道题。" + (f" 已保存：{saved_file.name}" if save_results.value else "")
    except (OSError, RuntimeError, ValueError) as error:
        status.value = f"第 {number} 题出错：{error}"
    finally:
        for control in controls:
            control.disabled = False

for control in (left, right, mode):
    control.observe(reset_comparison, names="value")
ask.on_click(submit_comparison)
clear.on_click(reset_comparison)
release_gpu.on_click(release_comparison)
reset_comparison()
status.value = "请输入问题，首次提问时加载模型。"
columns = [widgets.VBox([title[index], answers[index]], layout=widgets.Layout(width="49%"))
           for index in range(2)]
display(widgets.VBox([
    widgets.HBox([left, right]), mode, questions,
    widgets.HBox([ask, clear, release_gpu, save_results]), status,
    widgets.HBox(columns, layout=widgets.Layout(width="100%", justify_content="space-between")),
]))


## 文件位置与恢复

```text
MyDrive/Yukino-Companion/
  source/training-source-distill-v3.zip       # 固定代码+原创标签+出处
  source/distill-v3-<SHA>/                   # 可直接查看的完整源码
  data/epubs/                               # 已上传小说
  models/qwen2.5-1.5b-instruct-989aa798/      # 固定基座与分词器
  retrain/epub-a100-v3-distill/
    inputs.json                             # 本轮输入/参数锁定
    initial/                                # 继续训练前的权重快照
    prepared/                               # train/val/test/出处/清单
    distill-01-config.json
    distill-01/
      checkpoint-*/                         # 每轮完整断点及完成标记
      adapter/                              # 最佳验证轮次 LoRA
      baseline.json
      training_report.json
    evaluation/distill-01/                  # 回归和保留题对照报告
    distill-01-gguf/                        # 最终候选离线文件
    comparison/                             # 用户勾选保存的问答
    *.log
```

断线：相同 A100 环境运行 1–4，自动恢复或读取已完成结果；5 会补完评测。
如想追加新的失败案例，在源码 `distill-data/seeds.jsonl` 中编写有出处的正确答案，
人工检查后重新打包并使用新 RUN_ID。不要把待改正的模型回答直接当训练目标。